In [4]:
!ls -lah /kaggle/input

total 12K
drwxr-xr-x 3 root root 4.0K Oct  8 05:55 .
drwxr-xr-x 5 root root 4.0K Oct  8 05:50 ..
drwxr-xr-x 3 root root 4.0K Oct  8 05:55 datasets


In [5]:
!find /kaggle/input/datasets -maxdepth 6 -type f -name "config.yaml" -print

/kaggle/input/datasets/debasishmohanty87/quantization/quant-benchmark/config.yaml


In [6]:
!rm -rf /kaggle/working/quant-benchmark

!cp -r /kaggle/input/datasets/debasishmohanty87/quantization/quant-benchmark \
      /kaggle/working/quant-benchmark

%cd /kaggle/working/quant-benchmark

!ls -la

/kaggle/working/quant-benchmark
total 32
drwxr-xr-x 4 root root 4096 Oct  8 05:56 .
drwxr-xr-x 4 root root 4096 Oct  8 05:56 ..
drwxr-xr-x 2 root root 4096 Oct  8 05:56 benchmark
-rw-r--r-- 1 root root 1216 Oct  8 05:56 config.yaml
drwxr-xr-x 2 root root 4096 Oct  8 05:56 profiling
-rw-r--r-- 1 root root 2205 Oct  8 05:56 README.md
-rw-r--r-- 1 root root   74 Oct  8 05:56 requirements.txt
-rw-r--r-- 1 root root 3841 Oct  8 05:56 run_benchmark.py


In [7]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

!nvidia-smi

Thu Oct  8 05:56:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [8]:
!pip install -q -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 41.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.7/33.7 MB 61.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 426.0/426.0 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 63.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.6/782.6 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 56.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 MB 43.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 729.6/729.6 kB 29.8 MB/s eta 0:00:00
   ━━━

In [9]:
!cat config.yaml

project:
  name: quantization-performance-benchmark
  version: "1.0"

model:
  served_name: quant-benchmark-model

server:
  host: 127.0.0.1
  port: 8000
  gpu_memory_utilization: 0.90
  max_model_len: 4096
  startup_timeout_seconds: 900
  enforce_eager: false

benchmark:
  input_tokens: 512
  output_tokens: 256
  num_prompts: 100
  warmup_prompts: 10

  concurrency_levels:
    - 1
    - 4
    - 8
    - 16
    - 32

  request_rate: inf

  percentile_metrics:
    - ttft
    - tpot
    - itl
    - e2el

  percentiles:
    - 50
    - 95
    - 99

  seed: 42
  temperature: 0

  # Number of independent benchmark repetitions.
  # 1 is the default full run; increase to 2-3 for statistical confidence.
  repetitions: 1

accuracy:
  enabled: true
  minimum_score: 0.80
  num_prompts: 20
  max_tokens: 128

cost:
  gpu_hourly_usd: 0.35

quantization:
  methods:
    - name: fp16
      model: Qwen/Qwen2.5-3B-Instruct
      quantization: null
      dtype: float16

    - name: int8
      model: Qwen/Qw

In [12]:
!pip uninstall -y torchaudio

Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128


In [13]:
!python -c "import torch; print('Torch:', torch.__version__); print('CUDA:', torch.version.cuda); print('TorchAudio removed: OK')"

Torch: 2.13.0+cu130
CUDA: 13.0
TorchAudio removed: OK


In [14]:
!python -c "import vllm; print('vLLM:', vllm.__version__)"

vLLM: 0.31.0


In [19]:
%cd /kaggle/working/quant-benchmark
!python run_benchmark.py

/kaggle/working/quant-benchmark
PRODUCTION-STYLE QUANTIZATION PERFORMANCE BENCHMARK
{
  "timestamp_utc": "2026-10-08T06:12:46.961998+00:00",
  "python": "3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]",
  "platform": "Linux-6.18.48+-x86_64-with-glibc2.39",
  "cuda_visible_devices": "0",
  "gpu": [
    {
      "error": "'str' object has no attribute 'decode'"
    }
  ],
  "vllm_version": "0.31.0",
  "torch_version": "2.13.0+cu130",
  "transformers_version": "5.16.1"
}

CONFIGURATION: fp16 | Qwen/Qwen2.5-3B-Instruct
Starting vLLM...
vLLM ready.
Accuracy smoke score: 90.0%

[fp16] repetition=1 concurrency=1
Running:
/usr/bin/python3 -m vllm.entrypoints.cli.main bench serve --backend openai-chat --base-url http://127.0.0.1:8000 --endpoint /v1/chat/completions --model quant-benchmark-model --tokenizer Qwen/Qwen2.5-3B-Instruct --dataset-name random --random-input-len 512 --random-output-len 256 --num-prompts 100 --max-concurrency 1 --request-rate inf --num-warmups 10 --percentile-metrics

# Quantization Benchmark Results

The benchmark was completed successfully across **FP16, GPTQ INT8, and GPTQ INT4** using vLLM on a Tesla T4 GPU.

All **15 benchmark configurations completed successfully with 0 failed requests**.

## Throughput

Output throughput is measured in generated tokens per second.

| Quantization | C=1 | C=4 | C=8 | C=16 | C=32 |
|---|---:|---:|---:|---:|---:|
| **FP16** | 22.03 | 119.97 | 209.49 | 332.77 | 447.89 |
| **INT8** | 54.29 | 200.14 | 335.21 | 492.15 | 633.49 |
| **INT4** | **71.53** | **261.59** | **422.94** | **583.74** | **710.76** |

### Key finding

**INT4 achieved the highest throughput at every concurrency level.**

At concurrency 32:

- FP16: **447.89 tok/s**
- INT8: **633.49 tok/s**
- INT4: **710.76 tok/s**

Compared with FP16:

- INT8 is **41.4% faster**
- INT4 is **58.7% faster**

Compared with INT8, INT4 is **12.2% faster** at concurrency 32.

## Latency

| Concurrency | Quantization | P50 TTFT | P95 TTFT | P50 TPOT | P95 E2E |
|---:|---|---:|---:|---:|---:|
| 1 | FP16 | 242.40 ms | 247.32 ms | 44.88 ms | 11.81 s |
| 1 | INT8 | 259.98 ms | 263.94 ms | 17.48 ms | 4.74 s |
| 1 | INT4 | **232.82 ms** | **235.66 ms** | **13.13 ms** | **3.60 s** |
| 4 | FP16 | 97.48 ms | 100.10 ms | 33.10 ms | 8.66 s |
| 4 | INT8 | 71.17 ms | 76.52 ms | 19.75 ms | 5.21 s |
| 4 | INT4 | **66.27 ms** | **73.64 ms** | **15.05 ms** | **4.02 s** |
| 8 | FP16 | 130.78 ms | 155.11 ms | 36.58 ms | 9.61 s |
| 8 | INT8 | 94.89 ms | 126.13 ms | 22.77 ms | 6.07 s |
| 8 | INT4 | **82.73 ms** | **117.42 ms** | **18.10 ms** | **4.84 s** |
| 16 | FP16 | 200.73 ms | 242.98 ms | 43.95 ms | 11.62 s |
| 16 | INT8 | 157.84 ms | 215.77 ms | 30.03 ms | 7.96 s |
| 16 | INT4 | **134.72 ms** | **194.75 ms** | **25.68 ms** | **6.78 s** |
| 32 | FP16 | 250.60 ms | 370.09 ms | 62.66 ms | 16.42 s |
| 32 | INT8 | 245.54 ms | 354.85 ms | 44.92 ms | 11.98 s |
| 32 | INT4 | **238.84 ms** | **328.13 ms** | **41.08 ms** | **10.93 s** |

## VRAM Usage

| Quantization | Peak VRAM |
|---|---:|
| FP16 | 13,975.2 MB |
| INT8 | 13,543.2 MB |
| INT4 | 13,543.2 MB |

The measured peak VRAM difference between FP16 and the GPTQ configurations was approximately **432 MB**.

> **Note:** These measurements represent observed vLLM/runtime GPU memory usage, not the raw size of model weights. Runtime allocation, KV cache, CUDA overhead, and other serving components contribute to the measured value.

## Reliability

| Quantization | Configurations | Failed Requests |
|---|---:|---:|
| FP16 | 5 | **0** |
| INT8 | 5 | **0** |
| INT4 | 5 | **0** |

All **15 configurations completed successfully** with no failed requests.

## Overall Conclusion

The benchmark demonstrates a clear performance advantage from quantization for this serving workload.

At concurrency 32, **GPTQ INT4 reached 710.76 generated tokens/sec**, compared with **447.89 tokens/sec for FP16**, representing approximately a **58.7% throughput improvement**.

INT4 also produced the lowest measured:

- P50 TTFT
- P95 TTFT
- P50 TPOT
- P95 end-to-end latency

among the three configurations at concurrency 32.

The benchmark therefore shows that, for this workload and hardware configuration, **INT4 provides the best overall serving performance**, followed by INT8 and then FP16.

### Benchmark Environment

- **Model:** Qwen/Qwen2.5-3B-Instruct
- **Serving engine:** vLLM 0.31.0
- **GPU:** NVIDIA Tesla T4
- **PyTorch:** 2.13.0+cu130
- **CUDA:** 13.0
- **Transformers:** 5.16.1
- **Benchmark concurrency:** 1, 4, 8, 16, 32
- **Requests per configuration:** 100
- **Failed requests:** 0

In [20]:
import shutil
from pathlib import Path

results_dir = Path("/kaggle/working/quant-benchmark/results")
zip_path = Path("/kaggle/working/quant-benchmark-results")

shutil.make_archive(
    str(zip_path),
    "zip",
    root_dir=results_dir
)

print(f"Created: {zip_path}.zip")
print(f"Size: {Path(str(zip_path) + '.zip').stat().st_size / 1024**2:.2f} MB")

Created: /kaggle/working/quant-benchmark-results.zip
Size: 0.50 MB


In [3]:
%cd /kaggle/working/quant-benchmark

!echo "===== PROJECT ====="
!find . -maxdepth 2 -type f | sort

!echo ""
!echo "===== CONFIG ====="
!sed -n '1,220p' config.yaml


!ls -la


%cd /kaggle/working/quant-benchmark

!ls -la

[Errno 2] No such file or directory: '/kaggle/working/quant-benchmark'
/kaggle/working
===== PROJECT =====
./.virtual_documents/__notebook_source__.ipynb

===== CONFIG =====
sed: can't read config.yaml: No such file or directory
total 12
drwxr-xr-x 3 root root 4096 Oct  8 09:41 .
drwxr-xr-x 5 root root 4096 Oct  8 09:41 ..
drwxr-xr-x 2 root root 4096 Oct  8 09:41 .virtual_documents
[Errno 2] No such file or directory: '/kaggle/working/quant-benchmark'
/kaggle/working
total 12
drwxr-xr-x 3 root root 4096 Oct  8 09:41 .
drwxr-xr-x 5 root root 4096 Oct  8 09:41 ..
drwxr-xr-x 2 root root 4096 Oct  8 09:41 .virtual_documents


In [4]:
from pathlib import Path
import shutil

src = Path("/kaggle/input/datasets/debasishmohanty87/quantization/quant-benchmark")
dst = Path("/kaggle/working/quant-benchmark")

print("Source exists:", src.exists())

if not src.exists():
    raise FileNotFoundError(f"Project dataset not found: {src}")

if dst.exists():
    shutil.rmtree(dst)

shutil.copytree(src, dst)

print("Project copied successfully.")
print("Location:", dst)

Source exists: True
Project copied successfully.
Location: /kaggle/working/quant-benchmark


In [5]:
%cd /kaggle/working/quant-benchmark

!echo "===== PROJECT FILES ====="
!find . -maxdepth 2 -type f | sort

!echo ""
!echo "===== CONFIG ====="
!sed -n '1,220p' config.yaml

/kaggle/working/quant-benchmark
===== PROJECT FILES =====
./benchmark/accuracy.py
./benchmark/__init__.py
./benchmark/memory.py
./benchmark/results.py
./benchmark/server.py
./benchmark/system.py
./benchmark/vllm_bench.py
./config.yaml
./profiling/README.md
./README.md
./requirements.txt
./run_benchmark.py

===== CONFIG =====
project:
  name: quantization-performance-benchmark
  version: "1.0"

model:
  served_name: quant-benchmark-model

server:
  host: 127.0.0.1
  port: 8000
  gpu_memory_utilization: 0.90
  max_model_len: 4096
  startup_timeout_seconds: 900
  enforce_eager: false

benchmark:
  input_tokens: 512
  output_tokens: 256
  num_prompts: 100
  warmup_prompts: 10

  concurrency_levels:
    - 1
    - 4
    - 8
    - 16
    - 32

  request_rate: inf

  percentile_metrics:
    - ttft
    - tpot
    - itl
    - e2el

  percentiles:
    - 50
    - 95
    - 99

  seed: 42
  temperature: 0

  # Number of independent benchmark repetitions.
  # 1 is the default full run; increase to 2-

In [6]:
%cd /kaggle/working/quant-benchmark

!sed -n '1,260p' benchmark/accuracy.py

/kaggle/working/quant-benchmark
import requests


def run_accuracy(cfg, method):
    """
    Lightweight API-level quality smoke test.
    It is intentionally deterministic and is not a replacement for a
    formal benchmark such as MMLU/GSM8K/HellaSwag.
    """
    if not cfg["accuracy"]["enabled"]:
        return {"score": None, "passed": True}

    base = f"http://{cfg['server']['host']}:{cfg['server']['port']}"
    model = cfg["model"]["served_name"]

    cases = [
        ("What is 2 + 2? Answer with only the number.", "4"),
        ("What is the capital of France? Answer with one word.", "Paris"),
        ("What is 10 * 7? Answer with only the number.", "70"),
        ("Name the planet known as the Red Planet.", "Mars"),
        ("How many days are in a leap year? Answer with only the number.", "366"),
        ("What is H2O commonly called? Answer with one word.", "water"),
        ("What is 15 - 6? Answer with only the number.", "9"),
        ("Which language is primarily used t

In [7]:
import importlib.util

print("datasets:", importlib.util.find_spec("datasets") is not None)
print("transformers:", importlib.util.find_spec("transformers") is not None)
print("requests:", importlib.util.find_spec("requests") is not None)

datasets: True
transformers: True
requests: True


In [10]:
%%writefile /kaggle/working/quant-benchmark/accuracy_benchmark.py

import json
import re
import time
from pathlib import Path

import requests
import yaml
from datasets import load_dataset


PROJECT_DIR = Path("/kaggle/working/quant-benchmark")
CONFIG_FILE = PROJECT_DIR / "config.yaml"
RESULTS_DIR = PROJECT_DIR / "results" / "accuracy"


def load_config():
    with open(CONFIG_FILE, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)


def normalize_answer(text):
    text = text.strip().lower()
    text = text.replace(",", "")
    text = re.sub(r"\s+", " ", text)
    return text


def extract_gsm8k_answer(answer):
    """
    GSM8K answers normally contain:
        #### 42

    Extract the final numeric answer.
    """
    matches = re.findall(r"####\s*(-?\d+(?:\.\d+)?)", answer)

    if matches:
        return normalize_answer(matches[-1])

    # Fallback: last number in the answer
    matches = re.findall(r"-?\d+(?:\.\d+)?", answer)
    if matches:
        return normalize_answer(matches[-1])

    return normalize_answer(answer)


def extract_model_answer(text):
    """
    Extract the final answer from model output.
    We ask the model to finish with:
        FINAL: <answer>
    """
    match = re.search(
        r"FINAL\s*:\s*(.+)",
        text,
        flags=re.IGNORECASE | re.DOTALL,
    )

    if match:
        answer = match.group(1).strip()
        answer = answer.split("\n")[0].strip()
        return normalize_answer(answer)

    return extract_gsm8k_answer(text)


def call_model(base_url, model, prompt, max_tokens=256):
    payload = {
        "model": model,
        "messages": [
            {
                "role": "user",
                "content": prompt,
            }
        ],
        "temperature": 0,
        "max_tokens": max_tokens,
    }

    response = requests.post(
        f"{base_url}/v1/chat/completions",
        json=payload,
        timeout=180,
    )

    response.raise_for_status()

    data = response.json()

    return data["choices"][0]["message"]["content"].strip()


def evaluate_gsm8k(cfg, method, num_samples=100):
    base_url = (
        f"http://{cfg['server']['host']}:{cfg['server']['port']}"
    )

    model = cfg["model"]["served_name"]

    print(f"\nLoading GSM8K...")
    dataset = load_dataset(
        "openai/gsm8k",
        "main",
        split="test",
    )

    num_samples = min(num_samples, len(dataset))

    correct = 0
    records = []

    start_time = time.time()

    for i in range(num_samples):
        item = dataset[i]

        question = item["question"]
        reference = item["answer"]

        reference_answer = extract_gsm8k_answer(reference)

        prompt = f"""
Solve the following math problem carefully.

Show your reasoning briefly, then put ONLY the final numeric answer
on the last line in this exact format:

FINAL: <number>

Problem:
{question}
""".strip()

        try:
            model_output = call_model(
                base_url,
                model,
                prompt,
                max_tokens=cfg["accuracy"]["max_tokens"],
            )

            predicted = extract_model_answer(model_output)

            is_correct = predicted == reference_answer

            if is_correct:
                correct += 1

            records.append(
                {
                    "index": i,
                    "question": question,
                    "reference_answer": reference_answer,
                    "predicted_answer": predicted,
                    "correct": is_correct,
                    "model_output": model_output,
                }
            )

        except Exception as exc:
            records.append(
                {
                    "index": i,
                    "question": question,
                    "reference_answer": reference_answer,
                    "predicted_answer": None,
                    "correct": False,
                    "error": str(exc),
                }
            )

        if (i + 1) % 10 == 0:
            print(
                f"{method['name']}: "
                f"{i + 1}/{num_samples} | "
                f"accuracy={correct / (i + 1):.2%}"
            )

    elapsed = time.time() - start_time
    accuracy = correct / num_samples

    result = {
        "benchmark": "gsm8k",
        "quantization": method["name"],
        "model": method["model"],
        "samples": num_samples,
        "correct": correct,
        "accuracy": accuracy,
        "accuracy_percent": round(accuracy * 100, 2),
        "elapsed_seconds": round(elapsed, 2),
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    }

    RESULTS_DIR.mkdir(parents=True, exist_ok=True)

    result_file = (
        RESULTS_DIR / f"gsm8k_{method['name']}.json"
    )

    with open(result_file, "w", encoding="utf-8") as f:
        json.dump(
            {
                "summary": result,
                "records": records,
            },
            f,
            indent=2,
        )

    print(
        f"\n{method['name']} GSM8K accuracy: "
        f"{accuracy:.2%}"
    )

    return result


def main():
    cfg = load_config()

    # Start with 100 examples.
    # Once the pipeline is verified, we can increase this.
    num_samples = 100

    all_results = []

    for method in cfg["quantization"]["methods"]:
        print("\n" + "=" * 70)
        print(f"METHOD: {method['name']}")
        print(f"MODEL:  {method['model']}")
        print("=" * 70)

        result = evaluate_gsm8k(
            cfg,
            method,
            num_samples=num_samples,
        )

        all_results.append(result)

    summary_file = RESULTS_DIR / "gsm8k_summary.json"

    with open(summary_file, "w", encoding="utf-8") as f:
        json.dump(
            all_results,
            f,
            indent=2,
        )

    print("\n" + "=" * 70)
    print("GSM8K ACCURACY SUMMARY")
    print("=" * 70)

    for result in all_results:
        print(
            f"{result['quantization']:>5} : "
            f"{result['accuracy_percent']:>6.2f}%"
        )


if __name__ == "__main__":
    main()

Overwriting /kaggle/working/quant-benchmark/accuracy_benchmark.py


In [11]:
%%writefile /kaggle/working/quant-benchmark/run_accuracy.py

import json
import sys
import time
from pathlib import Path

import yaml

from benchmark.server import VLLMServer
from accuracy_benchmark import evaluate_gsm8k


PROJECT_DIR = Path("/kaggle/working/quant-benchmark")
CONFIG_FILE = PROJECT_DIR / "config.yaml"
RESULTS_DIR = PROJECT_DIR / "results" / "accuracy"


def load_config():
    with open(CONFIG_FILE, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)


def main():
    cfg = load_config()

    RESULTS_DIR.mkdir(parents=True, exist_ok=True)

    all_results = []

    for method in cfg["quantization"]["methods"]:

        print("\n" + "=" * 80)
        print(f"STARTING ACCURACY EVALUATION: {method['name'].upper()}")
        print(f"MODEL: {method['model']}")
        print("=" * 80)

        server_log = (
            RESULTS_DIR / f"server_{method['name']}.log"
        )

        server = VLLMServer(
            cfg,
            method,
        )

        try:
            print("Starting vLLM server...")
            server.start(server_log)

            print("vLLM server is ready.")

            result = evaluate_gsm8k(
                cfg,
                method,
                num_samples=100,
            )

            all_results.append(result)

        except Exception as exc:
            print(
                f"\nERROR during {method['name']} evaluation:"
                f"\n{exc}"
            )

            all_results.append(
                {
                    "benchmark": "gsm8k",
                    "quantization": method["name"],
                    "model": method["model"],
                    "error": str(exc),
                }
            )

        finally:
            print(f"Stopping {method['name']} server...")
            server.stop()
            time.sleep(3)

    # ---------------------------------------------------------
    # Calculate degradation versus FP16
    # ---------------------------------------------------------

    valid_results = [
        r for r in all_results
        if "accuracy" in r
    ]

    fp16 = next(
        (
            r for r in valid_results
            if r["quantization"] == "fp16"
        ),
        None,
    )

    if fp16:
        baseline = fp16["accuracy"]

        for result in valid_results:
            result["accuracy_delta_vs_fp16"] = round(
                result["accuracy"] - baseline,
                6,
            )

            result["accuracy_delta_pp_vs_fp16"] = round(
                (result["accuracy"] - baseline) * 100,
                2,
            )

    summary_file = RESULTS_DIR / "gsm8k_final_summary.json"

    with open(summary_file, "w", encoding="utf-8") as f:
        json.dump(
            all_results,
            f,
            indent=2,
        )

    print("\n")
    print("=" * 80)
    print("FINAL GSM8K ACCURACY RESULTS")
    print("=" * 80)

    print(
        f"{'Model':<10}"
        f"{'Accuracy':>12}"
        f"{'Delta vs FP16':>18}"
    )

    print("-" * 80)

    for result in all_results:

        if "accuracy" not in result:
            print(
                f"{result['quantization']:<10}"
                f"{'FAILED':>12}"
            )
            continue

        delta = result.get(
            "accuracy_delta_pp_vs_fp16",
            0.0,
        )

        print(
            f"{result['quantization']:<10}"
            f"{result['accuracy_percent']:>10.2f}%"
            f"{delta:>15.2f} pp"
        )

    print("=" * 80)
    print(f"\nResults saved to: {RESULTS_DIR}")
    print(f"Summary: {summary_file}")


if __name__ == "__main__":
    main()

Overwriting /kaggle/working/quant-benchmark/run_accuracy.py


In [12]:
from datasets import load_dataset

print("Loading GSM8K test dataset...")

dataset = load_dataset(
    "openai/gsm8k",
    "main",
    split="test",
)

print("Dataset loaded successfully.")
print("Number of test examples:", len(dataset))
print("\nFirst example:")
print(dataset[0])

Loading GSM8K test dataset...


README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.31MB            

main/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

main/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  419kB            

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Dataset loaded successfully.
Number of test examples: 1319

First example:
{'question': "Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?", 'answer': 'Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.\nShe makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.\n#### 18'}


In [15]:
!pip install -q vllm==0.31.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.6/320.6 MB 6.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 64.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.7/33.7 MB 56.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 426.0/426.0 kB 10.7 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 71.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.6/782.6 kB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 60.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 MB 47.9 MB/s eta 0:00:00:00:01

In [17]:
import torch

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Torch: 2.13.0+cu130
CUDA available: True
CUDA version: 13.0
GPU: Tesla T4


In [18]:
%cd /kaggle/working/quant-benchmark

from pathlib import Path

path = Path("benchmark/server.py")
text = path.read_text()

text = text.replace(
    '        cmd = [\n            "vllm", "serve",',
    '        cmd = [\n            "python", "-m", "vllm.entrypoints.cli.main", "serve",'
)

path.write_text(text)

print("server.py updated.")

/kaggle/working/quant-benchmark
server.py updated.


In [19]:
%cd /kaggle/working/quant-benchmark

!python run_accuracy_test.py

/kaggle/working/quant-benchmark

STARTING ACCURACY EVALUATION: FP16
MODEL: Qwen/Qwen2.5-3B-Instruct
Starting vLLM server...

ERROR during fp16 evaluation:
vLLM exited during startup. See /kaggle/working/quant-benchmark/results/accuracy/server_fp16.log
Stopping fp16 server...

STARTING ACCURACY EVALUATION: INT8
MODEL: Qwen/Qwen2.5-3B-Instruct-GPTQ-Int8
Starting vLLM server...

ERROR during int8 evaluation:
vLLM exited during startup. See /kaggle/working/quant-benchmark/results/accuracy/server_int8.log
Stopping int8 server...

STARTING ACCURACY EVALUATION: INT4
MODEL: Qwen/Qwen2.5-3B-Instruct-GPTQ-Int4
Starting vLLM server...

ERROR during int4 evaluation:
vLLM exited during startup. See /kaggle/working/quant-benchmark/results/accuracy/server_int4.log
Stopping int4 server...


FINAL GSM8K ACCURACY RESULTS
Model         Accuracy     Delta vs FP16
--------------------------------------------------------------------------------
fp16            FAILED
int8            FAILED
int4            F

In [20]:
%cd /kaggle/working/quant-benchmark

print("===== FP16 LOG =====")
!tail -n 100 results/accuracy/server_fp16.log

print("\n===== INT8 LOG =====")
!tail -n 100 results/accuracy/server_int8.log

print("\n===== INT4 LOG =====")
!tail -n 100 results/accuracy/server_int4.log 2>/dev/null || true

/kaggle/working/quant-benchmark
===== FP16 LOG =====
Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.13/dist-packages/vllm/entrypoints/cli/main.py", line 124, in <module>
    main()
    ~~~~^^
  File "/usr/local/lib/python3.13/dist-packages/vllm/entrypoints/cli/main.py", line 43, in main
    import vllm.entrypoints.cli.benchmark.main
  File "/usr/local/lib/python3.13/dist-packages/vllm/entrypoints/cli/benchmark/main.py", line 12, in <module>
    from vllm.entrypoints.serve.utils.api_utils import VLLM_SUBCMD_PARSER_EPILOG
  File "/usr/local/lib/python3.13/dist-packages/vllm/entrypoints/serve/utils/api_utils.py", line 19, in <module>
    from vllm.engine.arg_utils import EngineArgs
  File "/usr/local/lib/python3.13/dist-packages/vllm/engine/arg_utils.py", line 37, in <module>
    from vllm.config import (
    ...<35 lines>...
    )
  File "/usr/local/lib/python3.13/d

In [21]:
!pip uninstall -y torchaudio

Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128


In [22]:
import importlib.util
import torch

print("Torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("torchaudio installed:", importlib.util.find_spec("torchaudio"))
print("vLLM installed:", importlib.util.find_spec("vllm"))

Torch: 2.13.0+cu130
CUDA: 13.0
torchaudio installed: None
vLLM installed: ModuleSpec(name='vllm', loader=<_frozen_importlib_external.SourceFileLoader object at 0x787d3308fa10>, origin='/usr/local/lib/python3.13/dist-packages/vllm/__init__.py', submodule_search_locations=['/usr/local/lib/python3.13/dist-packages/vllm'])


In [23]:
%cd /kaggle/working/quant-benchmark
!python run_accuracy_test.py

/kaggle/working/quant-benchmark

STARTING ACCURACY EVALUATION: FP16
MODEL: Qwen/Qwen2.5-3B-Instruct
Starting vLLM server...
vLLM server is ready.

Loading GSM8K...
fp16: 10/100 | accuracy=0.00%
fp16: 20/100 | accuracy=0.00%
fp16: 30/100 | accuracy=3.33%
fp16: 40/100 | accuracy=2.50%
fp16: 50/100 | accuracy=4.00%
fp16: 60/100 | accuracy=3.33%
fp16: 70/100 | accuracy=2.86%
fp16: 80/100 | accuracy=2.50%
fp16: 90/100 | accuracy=2.22%
fp16: 100/100 | accuracy=3.00%

fp16 GSM8K accuracy: 3.00%
Stopping fp16 server...

STARTING ACCURACY EVALUATION: INT8
MODEL: Qwen/Qwen2.5-3B-Instruct-GPTQ-Int8
Starting vLLM server...
vLLM server is ready.

Loading GSM8K...
int8: 10/100 | accuracy=0.00%
int8: 20/100 | accuracy=0.00%
int8: 30/100 | accuracy=0.00%
int8: 40/100 | accuracy=0.00%
int8: 50/100 | accuracy=2.00%
int8: 60/100 | accuracy=1.67%
int8: 70/100 | accuracy=1.43%
int8: 80/100 | accuracy=1.25%
int8: 90/100 | accuracy=1.11%
int8: 100/100 | accuracy=1.00%

int8 GSM8K accuracy: 1.00%
Stopping int

In [24]:
%cd /kaggle/working/quant-benchmark

import json

path = "results/accuracy/gsm8k_fp16.json"

with open(path, "r") as f:
    data = json.load(f)

print("Summary:")
print(data["summary"])

print("\n===== SAMPLE PREDICTIONS =====")

for row in data["records"][:10]:
    print("\nQuestion:", row["question"])
    print("Expected:", row["reference_answer"])
    print("Predicted:", row["predicted_answer"])
    print("Correct:", row["correct"])
    print("Model output:")
    print(row.get("model_output", "")[:1000])

/kaggle/working/quant-benchmark
Summary:
{'benchmark': 'gsm8k', 'quantization': 'fp16', 'model': 'Qwen/Qwen2.5-3B-Instruct', 'samples': 100, 'correct': 3, 'accuracy': 0.03, 'accuracy_percent': 3.0, 'elapsed_seconds': 545.96, 'timestamp': '2026-10-08T10:08:28Z'}

===== SAMPLE PREDICTIONS =====

Question: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?
Expected: 18
Predicted: 4
Correct: False
Model output:
To determine how much Janet makes every day at the farmers' market, we need to follow these steps:

1. **Calculate the total number of eggs laid per day:**
   \[
   \text{Total eggs per day} = 16
   \]

2. **Determine the number of eggs eaten for breakfast:**
   \[
   \text{Eggs eaten for breakfast} = 3
   \]

3. **Calculate the number of eggs used for baki

In [25]:
%cd /kaggle/working/quant-benchmark

from pathlib import Path

path = Path("accuracy_benchmark.py")
text = path.read_text()

text = text.replace(
    "max_tokens=cfg[\"accuracy\"][\"max_tokens\"],",
    "max_tokens=512,"
)

path.write_text(text)

print("Updated GSM8K max_tokens to 512.")

/kaggle/working/quant-benchmark
Updated GSM8K max_tokens to 512.


In [26]:
%cd /kaggle/working/quant-benchmark

from pathlib import Path

path = Path("accuracy_benchmark.py")
text = path.read_text()

old = """Solve the following math problem carefully.

Show your reasoning briefly, then put ONLY the final numeric answer
on the last line in this exact format:

FINAL: <number>

Problem:
{question}"""

new = """Solve this math problem.

Give a concise calculation and finish with the final numeric answer
in this exact format:

FINAL: <number>

Problem:
{question}"""

if old not in text:
    print("Prompt block not found. Check file before changing anything.")
else:
    text = text.replace(old, new)
    path.write_text(text)
    print("Prompt updated successfully.")

/kaggle/working/quant-benchmark
Prompt updated successfully.


In [27]:
!grep -n -A12 "prompt = f" accuracy_benchmark.py
!grep -n "max_tokens" accuracy_benchmark.py

124:        prompt = f"""
125-Solve this math problem.
126-
127-Give a concise calculation and finish with the final numeric answer
128-in this exact format:
129-
130-FINAL: <number>
131-
132-Problem:
133-{question}
134-""".strip()
135-
136-        try:
69:def call_model(base_url, model, prompt, max_tokens=256):
79:        "max_tokens": max_tokens,
141:                max_tokens=512,


In [28]:
%cd /kaggle/working/quant-benchmark

!sed 's/num_samples = 100/num_samples = 10/' run_accuracy.py > run_accuracy_10.py

!python run_accuracy_10.py

/kaggle/working/quant-benchmark

STARTING ACCURACY EVALUATION: FP16
MODEL: Qwen/Qwen2.5-3B-Instruct
Starting vLLM server...
vLLM server is ready.

Loading GSM8K...
fp16: 10/100 | accuracy=60.00%
fp16: 20/100 | accuracy=70.00%
fp16: 30/100 | accuracy=70.00%
fp16: 40/100 | accuracy=72.50%
fp16: 50/100 | accuracy=72.00%
fp16: 60/100 | accuracy=73.33%
fp16: 70/100 | accuracy=74.29%
fp16: 80/100 | accuracy=75.00%
fp16: 90/100 | accuracy=74.44%
fp16: 100/100 | accuracy=75.00%

fp16 GSM8K accuracy: 75.00%
Stopping fp16 server...

STARTING ACCURACY EVALUATION: INT8
MODEL: Qwen/Qwen2.5-3B-Instruct-GPTQ-Int8
Starting vLLM server...
vLLM server is ready.

Loading GSM8K...
int8: 10/100 | accuracy=80.00%
int8: 20/100 | accuracy=70.00%
int8: 30/100 | accuracy=73.33%
int8: 40/100 | accuracy=75.00%
int8: 50/100 | accuracy=78.00%
int8: 60/100 | accuracy=78.33%
int8: 70/100 | accuracy=78.57%
int8: 80/100 | accuracy=78.75%
int8: 90/100 | accuracy=77.78%
int8: 100/100 | accuracy=78.00%

int8 GSM8K accurac

In [29]:
%cd /kaggle/working/quant-benchmark

!zip -r /kaggle/working/quant-benchmark-accuracy-results.zip results/accuracy

from IPython.display import FileLink, display

display(FileLink(
    "/kaggle/working/quant-benchmark-accuracy-results.zip",
    result_html_prefix="⬇️ Download Accuracy Results: "
))

/kaggle/working/quant-benchmark
  adding: results/accuracy/ (stored 0%)
  adding: results/accuracy/gsm8k_int8.json (deflated 75%)
  adding: results/accuracy/gsm8k_final_summary.json (deflated 72%)
  adding: results/accuracy/gsm8k_fp16.json (deflated 75%)
  adding: results/accuracy/gsm8k_int4.json (deflated 75%)
  adding: results/accuracy/server_int8.log (deflated 82%)
  adding: results/accuracy/server_int4.log (deflated 82%)
  adding: results/accuracy/server_fp16.log (deflated 85%)


/kaggle/working/quant-benchmark-accuracy-results.zip